# Stage 1: Per-Wallet Copy Sizing (tier3@2-0)

Fit the wallet universe and per-wallet copy weights ``alpha_w`` on **all data
resolved before August** (the full 2026 history), pick sizing hyperparameters
on **June+July** by sim Sharpe, single **test** pass on **August**.
Copy qty is capped by the reconstructed share-depth ``bucket_avail_copy_qty``.

**Output:** `stage1_scaled_result.json` + `signal_lab/wallet_scaling_{sim,ci,contrib}.csv`



In [1]:
# Setup: imports, paths, constants
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

NB_DIR = Path.cwd() if "__file__" not in globals() else Path(__file__).resolve().parent
sys.path.insert(0, str(NB_DIR))
OUT_DIR = NB_DIR / "signal_lab"

import numpy as np
import pandas as pd

from lib import DEFAULT_TAGS
from signal_lab.filters import COPY_DEFAULT, STRATEGY_SELECTION
from signal_lab.signal_lib import spearman_rho
from signal_lab.sizing import (
    block_bootstrap_sharpe,
    capital_constrained_sim,
    sizing_sharpe,
)
from signal_lab.stage1 import (
    attach_copy_wallet_metrics,
    candidate_splits_for,
    load_stage1_data,
)
from signal_lab.wallet_scaling import (
    alpha_kelly,
    alpha_tier,
    attach_depth_cap,
    run_sim,
    sim_row,
    wallet_daily_pnl,
    wallet_stats,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

BUDGET = 10_000.0
EXPOSURE_BUDGET = 10000.0  # per-token (condition_id + token_id) capital cap
MAX_LEAD_DAYS = 2  # keep only trades within this many days of contract resolution
ALPHA_MAX_GRID = (1,)
TIER_GRID = [(nt, am, amin) for nt in (3,) for am in ALPHA_MAX_GRID for amin in (0.0, 0.25)]
UNIFORM_K_GRID = (1.0,)

# Wallet filter: COPY_DEFAULT or STRATEGY_SELECTION
COPY_FILTER = STRATEGY_SELECTION

# Headline strategy for the exposure plot + saved result:
#   "kelly" | "tier" | "uniform" | "copy_all"  -> that scheme's scale-chosen config
#   "best"                                     -> max scale-window Sharpe overall
SAVE_STRATEGY = "best"

# PnL variant: (pnl_col, qty_col)
PNL_VARIANT = ("copyable_pnl", "copyable_qty_5m_100")
# PNL_VARIANT = ("copyable_pnl_20m_100", "copyable_qty_20m_100")

PNL_COL, QTY_COL = PNL_VARIANT
AVAIL_COL = QTY_COL.replace("copyable_qty_", "avail_copy_qty_")

# Windows (markets bucketed by last_condition_trade_ts):
#   fit   = everything resolved before Aug ("the whole 2026") -> wallet
#           selection (cell below recomputes metrics on this window) + alphas
#   scale = June + July -> sizing scheme / hyperparameter grid search
#   test  = August -> single honest pass
SPLIT = {"train_end": "2026-06-01", "val_end": "2026-08-01", "test_start": "2026-08-01"}


## Load data

In [2]:
df_full, df_train, df_val, df_test, wallet_metrics, hold_metrics = load_stage1_data(tags=DEFAULT_TAGS, **SPLIT, max_lead_days=MAX_LEAD_DAYS)
print(f"df_full: {len(df_full):,}")
print(f"  train: {len(df_train):,}  val: {len(df_val):,}  test: {len(df_test):,}")


Markets: 2870515
Filtered markets for {'Weather'}: 118970
Loading 16 trade shards...
Total trades loaded: 19,859,926
Unique wallets: 4,814
Date range: 2025-01-09 15:32:39+00:00 -> 2026-08-31 06:06:46+00:00
Lead filter (<= 2d before resolution): 18,631,434 trades
split_data_at_dates: train_end=2026-06-01 val_end=2026-08-01 test_start=2026-08-01
  Train:  4,326,085 trades  (21,773 markets)
  Val:    7,434,133 trades  (37,614 markets)
  Test:   2,321,121 trades  (22,959 markets)
  Total: 14,081,339 trades  (82,346 markets)
df_full: 14,081,339
  train: 4,326,085  val: 7,434,133  test: 2,321,121


## Copy universe

Candidate wallets = `COPY_FILTER`, fitted on the fit window (all data resolved
before August).



In [3]:
df_fit = pd.concat([df_train, df_val])  # BUY stream resolved before August
fit_metrics = attach_copy_wallet_metrics(df_fit)
del df_fit
wallets = set(COPY_FILTER(fit_metrics, hold_metrics))
print(f"{COPY_FILTER.name} wallets (fit < {SPLIT['val_end']}): {len(wallets)}")


strategy_selection wallets (fit < 2026-08-01): 16


In [4]:
# fit_metrics[fit_metrics['wallet'] == '0x7c63520c2ca9b336af0c205b9ccf68217bb393d4'] #[['buy_copyable_pnl', 'buy_copyable_roi','estimated_copyable_buy_sharpe','max_copyable_drawdown_to_copyable_pnl', 'copyable_pnl_similarity'  ]]

## Share-depth cap

Cap = stage0 Phase 2's per-bucket max copy quantity (`avail_copy_qty_5m_100`), exported with the processed trades.

In [5]:
splits = candidate_splits_for(df_full, wallets, **SPLIT)
splits = attach_depth_cap(splits, avail_col=AVAIL_COL, qty_col=QTY_COL)
for fr in splits.values():
    fr["token_group"] = fr["condition_id"] + "_" + fr["token_id"]
del df_full, df_train, df_val, df_test

for name in ("train", "val", "test"):
    fr = splits[name]
    capped = (fr["bucket_avail_copy_qty"] < fr[QTY_COL]).mean()
    print(f"{name:5s}: {len(fr):,}  trades_capped_by_depth={capped:.3f}")


split_data_at_dates: train_end=2026-06-01 val_end=2026-08-01 test_start=2026-08-01
  Train:     13,232 trades  (3,052 markets)
  Val:       27,395 trades  (5,819 markets)
  Test:       7,585 trades  (2,282 markets)
  Total:     48,212 trades  (11,153 markets)
train: 13,232  trades_capped_by_depth=0.000
val  : 27,395  trades_capped_by_depth=0.000
test : 7,585  trades_capped_by_depth=0.000


## Fit-window per-wallet stats

Per-wallet daily pnl (copyable, alpha=1) over the full pre-August candidate
history with mean/std shrinkage -> Sharpe proxy.



In [6]:
pre_aug = pd.concat([splits["train"], splits["val"]])  # resolved before Aug
fit_daily = wallet_daily_pnl(pre_aug, pnl_col=PNL_COL)
st = wallet_stats(fit_daily, pnl_col=PNL_COL)
print(f"wallets with fit-window daily series: {len(st)}")
st[["mu", "sigma", "n_days", "sharpe_proxy", "total_pnl"]].sort_values(
    "sharpe_proxy", ascending=False
).head(15)


wallets with fit-window daily series: 16


,mu,sigma,n_days,sharpe_proxy,total_pnl
wallet,,,,,
0x5b0a52782a38f1318c374317fbd9cf90b7d74d57,7.4448,23.7184,164,0.2712,1220.9531
0x1f127e31717c8d74232fc4d5693e9236b49c9424,4.1163,19.2991,417,0.1980,1716.5135
0xaa05cc99fa6913d1f39c517be218cdd2ebcb9907,2.4015,11.7695,247,0.1590,593.1611
0x5fe79a73af0a8ad6a83d89256026b14f35981ea9,14.9560,100.7450,348,0.1455,5204.6755
0xa89518aca5a633a79ad1e9737209c9689f83faac,10.5465,72.4770,325,0.1426,3427.6043
0xb9012e0d9b60d3920286309328b935cdfa609fc4,2.6755,19.1442,973,0.1362,2603.2889
0xf10e941dbd215b98b90be7dd9ec61da409ef4308,1.7849,13.3037,531,0.1225,947.7892
0x331d8ec8d8931b479007f4931a6b25b0b6628846,4.4727,38.8366,517,0.1139,2312.3886
0x96e60b753df1367cb3ee81fbb02df92bc61afece,1.5718,12.6951,616,0.1139,968.2492


## Weight schemes

All benchmarked vs copy-all: shrunk max-Sharpe (Kelly), tier, uniform-k.

In [7]:
schemes = {}
for am in ALPHA_MAX_GRID:
    schemes[f"kelly@{am:g}"] = ("kelly", alpha_kelly(st, am), {"alpha_max": am})
for (nt, am, amin) in TIER_GRID:
    schemes[f"tier{nt}@{am:g}-{amin:g}"] = (
        "tier",
        alpha_tier(st, nt, am, amin),
        {"n_tiers": nt, "alpha_max": am, "alpha_min": amin},
    )
for k in UNIFORM_K_GRID:
    schemes[f"uniform@{k:g}"] = ("uniform", pd.Series(k, index=st.index), {"k": k})
schemes["copy_all"] = ("copy_all", pd.Series(1.0, index=st.index), {})

print(f"schemes: {len(schemes)}")


schemes: 5


## Scheme selection (June+July)

Objective: annualized Sharpe of daily resolution-pnl, $1000 per-token exposure
cap. Best config per scheme carries into the single August test pass.



In [8]:
sim_rows = []
best_per_scheme = {}
for name, (scheme, alpha_map, params) in schemes.items():
    res = run_sim(splits["val"], alpha_map, pnl_col=PNL_COL, qty_col=QTY_COL,
                 group_col="token_group", group_budget=EXPOSURE_BUDGET)
    row = sim_row(scheme, name, "scale", res)
    sim_rows.append(row)
    key = scheme if scheme != "kelly" else "kelly"
    if key not in best_per_scheme or row["sharpe_daily"] > best_per_scheme[key][2]:
        best_per_scheme[key] = (name, params, row["sharpe_daily"])

sim_df = pd.DataFrame(sim_rows)
sim_df[sim_df["split"] == "scale"].sort_values("sharpe_daily", ascending=False).head(15)


,scheme,config,split,trades,pnl,roi_w,sharpe_daily,mean_used,peak_used
0,kelly,kelly@1,scale,19580,15701.0600,0.1160,2.4650,733.8100,3549.4900
1,tier,tier3@1-0,scale,14364,22419.5300,0.1319,2.0560,776.0400,4798.9300
2,tier,tier3@1-0.25,scale,19580,23433.3700,0.1339,1.8430,841.7000,4723.5300
3,uniform,uniform@1,scale,19580,24838.6200,0.1375,1.7640,948.3300,4768.5100
4,copy_all,copy_all,scale,19580,24838.6200,0.1375,1.7640,948.3300,4768.5100


In [9]:
print("Selected per scheme (by scale-window Sharpe):")
for key, (name, params, scale_sharpe) in best_per_scheme.items():
    print(f"  {key:10s} -> {name:>22s}  scale_sharpe={scale_sharpe:.3f}")

assert SAVE_STRATEGY == "best" or SAVE_STRATEGY in best_per_scheme, (
    f"SAVE_STRATEGY={SAVE_STRATEGY!r} not one of {sorted(best_per_scheme)} | 'best'"
)
if SAVE_STRATEGY == "best":
    best_name = max(best_per_scheme.values(), key=lambda x: x[2])[0]
else:
    best_name = best_per_scheme[SAVE_STRATEGY][0]
print(f"\nHeadline strategy ({SAVE_STRATEGY}): {best_name}")


Selected per scheme (by scale-window Sharpe):
  kelly      ->                kelly@1  scale_sharpe=2.465
  tier       ->              tier3@1-0  scale_sharpe=2.056
  uniform    ->              uniform@1  scale_sharpe=1.764
  copy_all   ->               copy_all  scale_sharpe=1.764

Headline strategy (best): kelly@1


## Test: single pass per chosen config

One honest test pass for each scheme's val-chosen config (10bps).

In [10]:
for key, (name, params, _scale_sharpe) in best_per_scheme.items():
    alpha_map = schemes[name][1]
    res = run_sim(splits["test"], alpha_map, pnl_col=PNL_COL, qty_col=QTY_COL,
                 group_col="token_group", group_budget=EXPOSURE_BUDGET)
    row = sim_row(schemes[name][0], name, "test", res)
    sim_rows.append(row)

sim_df = pd.DataFrame(sim_rows)
sim_df.to_csv(OUT_DIR / "wallet_scaling_sim.csv", index=False)
sim_df[sim_df["split"] == "test"].sort_values("sharpe_daily", ascending=False)


,scheme,config,split,trades,pnl,roi_w,sharpe_daily,mean_used,peak_used
6,tier,tier3@1-0,test,3712,3485.2800,0.0782,1.8080,332.5200,3244.7900
7,uniform,uniform@1,test,5118,3412.2400,0.0799,1.8000,341.9000,2465.6800
8,copy_all,copy_all,test,5118,3412.2400,0.0799,1.8000,341.9000,2465.6800
5,kelly,kelly@1,test,5118,2475.5800,0.0691,1.4650,268.4500,2578.0000


## Robustness: cost sweep + bootstrap CI

Cost sweep (0/10/30bps) + 7-day block-bootstrap Sharpe CI on test.

In [11]:
ci_rows = []
for key, (name, params, _) in best_per_scheme.items():
    alpha_map = schemes[name][1]
    res = run_sim(splits["test"], alpha_map, pnl_col=PNL_COL, qty_col=QTY_COL,
                 group_col="token_group", group_budget=EXPOSURE_BUDGET)
    point, lo, hi = block_bootstrap_sharpe(res["daily_pnl"], block_size=7, n_iter=1000, seed=42)
    ci_rows.append({
        "design": name,
        "pnl": round(res["net_pnl"], 2),
        "roi_w": round(res["net_pnl"] / res["notional"], 4) if res["notional"] > 0 else np.nan,
        "sharpe_daily": round(sizing_sharpe(res["daily_pnl"], 365.0), 3),
        "ci_lo": round(lo, 3), "ci_hi": round(hi, 3),
    })

res_all = capital_constrained_sim(splits["test"], "score1", float("inf"), 1.0,
                                  group_col="token_group", group_budget=EXPOSURE_BUDGET,
                                  pnl_col=PNL_COL, qty_col=QTY_COL)
point, lo, hi = block_bootstrap_sharpe(res_all["daily_pnl"], block_size=7, n_iter=1000, seed=42)
ci_rows.append({
    "design": "copy_all",
    "pnl": round(res_all["net_pnl"], 2),
    "roi_w": round(res_all["net_pnl"] / res_all["notional"], 4) if res_all["notional"] > 0 else np.nan,
    "sharpe_daily": round(sizing_sharpe(res_all["daily_pnl"], 365.0), 3),
    "ci_lo": round(lo, 3), "ci_hi": round(hi, 3),
})

ci_df = pd.DataFrame(ci_rows)
ci_df.to_csv(OUT_DIR / "wallet_scaling_ci.csv", index=False)
ci_df


,design,pnl,roi_w,sharpe_daily,ci_lo,ci_hi
0,kelly@1,2475.5800,0.0691,1.4650,-1.9590,8.9170
1,tier3@1-0,3485.2800,0.0782,1.8080,0.0160,8.1500
2,uniform@1,3412.2400,0.0799,1.8000,0.8120,8.7630
3,copy_all,3412.2400,0.0799,1.8000,0.8120,8.7630
4,copy_all,3412.2400,0.0799,1.8000,0.8120,8.7630


In [12]:
list(schemes['uniform@1'][1].index.values)

['0x04b3f873b003eba3774df1e5e47a370d27ee1b7e',
 '0x15e9a4f23c85ce0d5b2016cd029d9185e9b3dacd',
 '0x1f127e31717c8d74232fc4d5693e9236b49c9424',
 '0x331d8ec8d8931b479007f4931a6b25b0b6628846',
 '0x41f623e0baae41e385875a3759c474cf0622828e',
 '0x5b0a52782a38f1318c374317fbd9cf90b7d74d57',
 '0x5f612351a9e46afbe3f164473a7d77a69ec9840a',
 '0x5fe79a73af0a8ad6a83d89256026b14f35981ea9',
 '0x85c06787486a90e05ba1df96c034ef95c8cce01d',
 '0x919698b19427cbe6945b0dc823f2d9e126a4d934',
 '0x96e60b753df1367cb3ee81fbb02df92bc61afece',
 '0xa89518aca5a633a79ad1e9737209c9689f83faac',
 '0xaa05cc99fa6913d1f39c517be218cdd2ebcb9907',
 '0xb9012e0d9b60d3920286309328b935cdfa609fc4',
 '0xf10e941dbd215b98b90be7dd9ec61da409ef4308',
 '0xf1e18ec32b2f1e123bc098e3956e6fd00012c152']

## Test-period exposure & PnL over time

Exposure opens at each BUY (`qty = alpha_w * copyable_qty_5m_100` capped by `bucket_avail_copy_qty`, at `price`) and closes at contract resolution `last_condition_trade_ts` — only for contracts resolved within the test window, so unresolved exposure stays open. PnL shown twice: attributed at trade time (`dt`) and at contract resolution time (`last_condition_trade_ts`, resolved contracts only).

In [13]:
import plotly.graph_objects as go

test = splits["test"].copy()
alpha_map = schemes[best_name][1]
test["alpha_w"] = test["wallet"].map(alpha_map).fillna(1.0)
test["raw_copy_pnl"] = test[PNL_COL]
test["wallet_buy_pnl"] = test["pnl"]
test["res_ts"] = pd.to_datetime(test["last_condition_trade_ts"], utc=True, errors="coerce")

# Run sim to get taken mask — exposure/qty must respect the budget
res = run_sim(splits["test"], alpha_map, pnl_col=PNL_COL, qty_col=QTY_COL,
                 group_col="token_group", group_budget=EXPOSURE_BUDGET)
taken_idx = set(res["taken"].values)
test["taken"] = test.index.isin(taken_idx)
test["qty"] = np.clip(test["alpha_w"] * test[QTY_COL], 0.0, test["bucket_avail_copy_qty"])

# Per-trade sim PnL: only for taken trades
per_share = test[PNL_COL] / test[QTY_COL].replace(0, np.nan)
test["copy_pnl"] = np.where(test["taken"], per_share * test["qty"], 0.0)

taken = test[test["taken"]].copy()
print(f"taken trades: {len(taken):,} / {len(test):,}  sim PnL: {taken["copy_pnl"].sum():,.0f}")

window_end = test["dt"].max()
resolved = test["res_ts"] <= window_end
print(
    f"test trades: {len(test):,}  contracts: {test["condition_id"].nunique():,}  "
    f"resolved by {window_end:%Y-%m-%d}: {int(resolved.sum()):,} trades "
    f"({test.loc[resolved, "condition_id"].nunique():,} contracts)"
)
print(
    f"raw {PNL_COL} sum: {test[PNL_COL].sum():,.0f}  "
    f"wallet pnl sum: {test["wallet_buy_pnl"].sum():,.0f}  "
    f"sim copy_pnl sum: {taken["copy_pnl"].sum():,.0f}"
)

# Exposure (scaled): only from taken trades
taken_resolved = taken["res_ts"] <= window_end
open_ev = pd.DataFrame({
    "ev_dt": taken["dt"],
    "exposure_delta": taken["qty"] * taken["price"],
})
close_ev = pd.DataFrame({
    "ev_dt": taken.loc[taken_resolved, "res_ts"],
    "exposure_delta": -(taken.loc[taken_resolved, "qty"] * taken.loc[taken_resolved, "price"]),
})
events = (
    pd.concat([open_ev, close_ev], ignore_index=True)
    .sort_values("ev_dt")
    .reset_index(drop=True)
)
events["exposure"] = events["exposure_delta"].cumsum()

# Exposure (raw): all trades, unscaled
raw_ev = pd.DataFrame({
    "ev_dt": test["dt"],
    "exposure_delta": test[QTY_COL] * test["price"],
})
raw_close = pd.DataFrame({
    "ev_dt": test.loc[resolved, "res_ts"],
    "exposure_delta": -(test.loc[resolved, QTY_COL] * test.loc[resolved, "price"]),
})
raw_events = (
    pd.concat([raw_ev, raw_close], ignore_index=True)
    .sort_values("ev_dt")
    .reset_index(drop=True)
)
raw_events["exposure"] = raw_events["exposure_delta"].cumsum()

def _cum_pnl(dt_col, pnl_col):
    df = test[[dt_col, pnl_col]].rename(columns={dt_col: "ev_dt", pnl_col: "pnl"})
    df = df.sort_values("ev_dt").reset_index(drop=True)
    df["cum_pnl"] = df["pnl"].cumsum()
    return df

pnl_trade = _cum_pnl("dt", "copy_pnl")
pnl_res = _cum_pnl("res_ts", "copy_pnl")
pnl_raw_trade = _cum_pnl("dt", "raw_copy_pnl")
pnl_raw_res = _cum_pnl("res_ts", "raw_copy_pnl")
pnl_wallet_trade = _cum_pnl("dt", "wallet_buy_pnl")
pnl_wallet_res = _cum_pnl("res_ts", "wallet_buy_pnl")

fig = go.Figure()
fig.add_trace(go.Scatter(x=events["ev_dt"], y=events["exposure"], mode="lines",
    name="exposure (scaled)", line=dict(color="rgba(31,119,180,0.6)")))
fig.add_trace(go.Scatter(x=raw_events["ev_dt"], y=raw_events["exposure"], mode="lines",
    name="exposure (raw)", line=dict(dash="dash", color="rgba(31,119,180,0.6)")))
fig.add_trace(go.Scatter(
    x=pnl_trade["ev_dt"], y=pnl_trade["cum_pnl"], mode="lines",
    name=f"cum {PNL_COL} sized (trade time)",
))
fig.add_trace(go.Scatter(
    x=pnl_res["ev_dt"], y=pnl_res["cum_pnl"], mode="lines", line=dict(dash="dash"),
    name=f"cum {PNL_COL} sized (resolution time)",
))
fig.add_trace(go.Scatter(
    x=pnl_raw_trade["ev_dt"], y=pnl_raw_trade["cum_pnl"], mode="lines",
    name=f"cum {PNL_COL} raw (trade time)", line=dict(color="rgba(255,127,14,0.6)"),
))
fig.add_trace(go.Scatter(
    x=pnl_raw_res["ev_dt"], y=pnl_raw_res["cum_pnl"], mode="lines",
    name=f"cum {PNL_COL} raw (resolution time)", line=dict(dash="dash", color="rgba(255,127,14,0.6)"),
))
fig.add_trace(go.Scatter(
    x=pnl_wallet_trade["ev_dt"], y=pnl_wallet_trade["cum_pnl"], mode="lines",
    name="cum wallet buy pnl (trade time)", line=dict(color="rgba(44,160,28,0.6)"),
))
fig.add_trace(go.Scatter(
    x=pnl_wallet_res["ev_dt"], y=pnl_wallet_res["cum_pnl"], mode="lines",
    name="cum wallet buy pnl (resolution time)", line=dict(dash="dash", color="rgba(44,160,28,0.6)"),
))
fig.update_layout(
    title=f"Test-period exposure & PnL over time — {best_name}",
    xaxis_title="Time",
    yaxis_title="USDC",
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
)
fig.show()


taken trades: 5,118 / 7,585  sim PnL: 2,476
test trades: 7,585  contracts: 2,282  resolved by 2026-08-30: 7,552 trades (2,266 contracts)
raw copyable_pnl sum: 3,412  wallet pnl sum: 16,986  sim copy_pnl sum: 2,476


## Per-wallet contributions

Train alphas vs forward (test) wallet stats.

In [14]:
test_daily = wallet_daily_pnl(splits["test"], pnl_col=PNL_COL)
test_st = test_daily.groupby("wallet")[PNL_COL].agg(
    test_pnl="sum", test_n_days="size"
)
test_sharpe = test_daily.groupby("wallet")[PNL_COL].apply(
    lambda s: (s.mean() / s.std() * np.sqrt(365.0)) if s.std() > 0 and len(s) >= 2 else np.nan
).rename("test_sharpe")

contrib = st.join(test_st, how="outer").join(test_sharpe, how="outer").fillna(0.0)
contrib = contrib[contrib["test_n_days"] > 0]
alpha_cont = schemes[best_per_scheme["kelly"][0]][1]
alpha_tier_cont = schemes[best_per_scheme["tier"][0]][1]
contrib["alpha_kelly"] = contrib.index.map(alpha_cont).fillna(1.0)
contrib["alpha_tier"] = contrib.index.map(alpha_tier_cont).fillna(1.0)
contrib = contrib.reset_index()
contrib["test_roi"] = contrib["test_pnl"] / contrib["total_pnl"].replace(0, np.nan)
contrib.to_csv(OUT_DIR / "wallet_scaling_contrib.csv", index=False)

a = contrib["alpha_kelly"].to_numpy()
ts = contrib["test_sharpe"].to_numpy()
valid = np.isfinite(ts)
rho = spearman_rho(pd.Series(a[valid]), pd.Series(ts[valid])) if valid.sum() > 2 else np.nan
print(f"Spearman(alpha_kelly, wallet test sharpe) = {rho:.4f}  (n={int(valid.sum())})")
contrib[["wallet", "alpha_kelly", "alpha_tier", "test_pnl", "test_sharpe", "test_roi"]].head(15)


Spearman(alpha_kelly, wallet test sharpe) = -0.1099  (n=13)


,wallet,alpha_kelly,alpha_tier,test_pnl,test_sharpe,test_roi
0,0x04b3f873b003eba3774df1e5e47a370d27ee1b7e,1.3779,0.0000,215.2876,1.1298,0.1368
1,0x15e9a4f23c85ce0d5b2016cd029d9185e9b3dacd,1.0680,0.8889,189.9671,1.1818,0.2153
2,0x41f623e0baae41e385875a3759c474cf0622828e,0.2084,0.8889,-17.6091,-16.2043,-0.0024
3,0x5f612351a9e46afbe3f164473a7d77a69ec9840a,1.7126,0.0000,219.0671,1.2932,0.2368
4,0x5fe79a73af0a8ad6a83d89256026b14f35981ea9,0.2566,1.7778,1131.8940,4.2488,0.2175
5,0x85c06787486a90e05ba1df96c034ef95c8cce01d,0.2332,0.0000,3.7459,13.5093,0.0013
6,0x919698b19427cbe6945b0dc823f2d9e126a4d934,0.7785,0.8889,756.3873,1.6904,0.1768
7,0x96e60b753df1367cb3ee81fbb02df92bc61afece,1.3886,0.8889,44.9201,0.4329,0.0464
8,0xa89518aca5a633a79ad1e9737209c9689f83faac,0.3487,1.7778,252.4200,4.0971,0.0736
9,0xaa05cc99fa6913d1f39c517be218cdd2ebcb9907,1.7846,1.7778,-67.5754,-10.2499,-0.1139


## Save stage 1 result

In [15]:
import json
from datetime import datetime, timezone

best_params = schemes[best_name][2]

wallet_cols = [
    "wallet", "mu", "sigma", "n_days", "total_pnl", "sharpe_proxy",
    "alpha_kelly", "alpha_tier", "test_pnl", "test_n_days", "test_sharpe", "test_roi",
]
wallet_records = contrib[[c for c in wallet_cols if c in contrib.columns]].to_dict(orient="records")


def _convert(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


wallet_records = [{k: _convert(v) for k, v in w.items()} for w in wallet_records]

test_row = sim_df[(sim_df["split"] == "test") & (sim_df["config"] == best_name)].iloc[0]
copy_all_row = sim_df[(sim_df["split"] == "test") & (sim_df["config"] == "copy_all")].iloc[0]

metadata = {
    "type": "scaled_copy",
    "saved_strategy": SAVE_STRATEGY,
    "tags": sorted(DEFAULT_TAGS),
    "quote_ttl": 300,
    "run_timestamp": datetime.now(timezone.utc).isoformat(),
    "n_wallets_selected": int((contrib["alpha_tier"] > 0).sum()),
    "n_wallets_total": len(wallets),
    "split_sizes": {k: int(len(v)) for k, v in splits.items()},
}

payload = {
    "stage": 1,
    "best_params": {k: _convert(v) for k, v in best_params.items()},
    "best_scale_sharpe": float(max(best_per_scheme.values(), key=lambda x: x[2])[2]),
    "test_performance": {
        "config": best_name,
        "trades": int(test_row["trades"]),
        "pnl": float(test_row["pnl"]),
        "roi_w": float(test_row["roi_w"]),
        "sharpe_daily": float(test_row["sharpe_daily"]),
        "copy_all": {
            "trades": int(copy_all_row["trades"]),
            "pnl": float(copy_all_row["pnl"]),
            "roi_w": float(copy_all_row["roi_w"]),
            "sharpe_daily": float(copy_all_row["sharpe_daily"]),
        },
    },
    "metadata": metadata,
    "wallets": wallet_records,
}

out_path = NB_DIR / "stage1_scaled_result.json"
with open(out_path, "w") as f:
    json.dump(payload, f, indent=2)
print(f"Saved stage 1 scaled result -> {out_path.resolve()}")


Saved stage 1 scaled result -> /Users/vobornij/projects/polymarket/notebooks/wallet_selection/stage1_scaled_result.json


## Price-scaling fill experiment (exploratory)

Test a limit-price entry idea on a **sample** (~1k test contracts, copy-default wallets):
copy each candidate copy-wallet BUY at `limit = price * scale` for
`scale ∈ {1.0, 0.98, 0.95, 0.90}` and give the order a **5-minute window** to fill.

- **Fill rule:** filled iff within `(dt, dt+5min]` any trade on the same
  `(condition_id, token_id)` prints at `price <= limit` with a strictly greater timestamp.
- **Fill price:** exactly the limit price, so
  `pnl = copyable_pnl + copyable_qty_5m_100 * (price - limit)` (same formula/quantity as the
  original `copyable_pnl`); unfilled trades contribute 0.
- **Baseline:** `scale = 1.0` is the market-copy (fill immediately at `price`), so it
  must reproduce `sum(copyable_pnl)` on the sample.


In [16]:
from lib import DEFAULT_TRADES_DIR
from signal_lab.wallet_scaling import price_scale_fill_sim

rng = np.random.RandomState(42)
test_markets = np.sort(splits["test"]["condition_id"].unique())
n_sel = min(1000, len(test_markets))
sel_markets = rng.choice(test_markets, size=n_sel, replace=False)
signals = splits["test"][splits["test"]["condition_id"].isin(sel_markets)].copy()
signals = signals[signals["copyable_qty_5m_100"] > 0]
print(f"test markets: {len(test_markets):,}  sampled: {n_sel:,}")
print(f"candidate BUYs (copyable_qty_5m_100>0) on sample: {len(signals):,}")

_tape_cols = ["condition_id", "token_id", "dt", "avg_price"]
tape_parts = []
for f in sorted(DEFAULT_TRADES_DIR.glob("*.parquet")):
    tp = pd.read_parquet(f, columns=_tape_cols)
    tp = tp[tp["condition_id"].isin(sel_markets)]
    if not tp.empty:
        tape_parts.append(tp.rename(columns={"avg_price": "price"}))
tape = (
    pd.concat(tape_parts, ignore_index=True)
    if tape_parts
    else pd.DataFrame(columns=["condition_id", "token_id", "dt", "price"])
)
print(f"fill tape rows (sampled contracts, both sides): {len(tape):,}")


test markets: 2,282  sampled: 1,000
candidate BUYs (copyable_qty_5m_100>0) on sample: 2,285
fill tape rows (sampled contracts, both sides): 446,125


In [17]:
SCALES = (1.0, 0.98, 0.95, 0.90)
sim = price_scale_fill_sim(signals, tape, scales=SCALES, window_minutes=5.0)
base_pnl = float(signals["copyable_pnl"].sum())

summary = (
    sim.groupby("scale")
    .agg(signals=("filled", "size"), fills=("filled", "sum"),
         fill_rate=("filled", "mean"), pnl=("pnl", "sum"))
    .reset_index()
)
summary["pnl_pct_of_market"] = summary["pnl"] / base_pnl * 100 if base_pnl else np.nan
summary["delta_vs_market"] = summary["pnl"] - base_pnl
print(f"market-copy pnl (baseline = sum copyable_pnl): {base_pnl:,.2f}")
summary.round(2)


market-copy pnl (baseline = sum copyable_pnl): 1,471.72


,scale,signals,fills,fill_rate,pnl,pnl_pct_of_market,delta_vs_market
0,0.9000,2285,597,0.2600,1064.1600,72.3100,-407.5700
1,0.9500,2285,695,0.3000,1176.3900,79.9300,-295.3300
2,0.9800,2285,804,0.3500,1378.6700,93.6800,-93.0600
3,1.0000,2285,2285,1.0000,1471.7200,100.0000,0.0000


In [18]:
pw_pnl = sim.pivot_table(index="wallet", columns="scale", values="pnl", aggfunc="sum")
pw_fill = sim.pivot_table(index="wallet", columns="scale", values="filled", aggfunc="mean")
pw = pw_pnl.join(pw_fill.rename(columns={c: f"fill_{c:g}" for c in pw_fill.columns}))
pw = pw.reindex(pw[1.0].sort_values(ascending=False).index)
pw.round(1).head(15)


scale,0.9000,0.9500,0.9800,1.0000,fill_0.9,fill_0.95,fill_0.98,fill_1
wallet,,,,,,,,
0x5fe79a73af0a8ad6a83d89256026b14f35981ea9,541.7000,727.3000,753.9000,516.1000,0.3000,0.3000,0.4000,1.0000
0x919698b19427cbe6945b0dc823f2d9e126a4d934,313.9000,360.1000,461.8000,496.8000,0.3000,0.4000,0.4000,1.0000
0x15e9a4f23c85ce0d5b2016cd029d9185e9b3dacd,57.2000,53.8000,82.5000,159.3000,0.4000,0.5000,0.5000,1.0000
0xa89518aca5a633a79ad1e9737209c9689f83faac,1.6000,5.9000,4.2000,126.9000,0.0000,0.1000,0.1000,1.0000
0xb9012e0d9b60d3920286309328b935cdfa609fc4,182.4000,106.3000,71.6000,95.1000,0.0000,0.1000,0.1000,1.0000
0x96e60b753df1367cb3ee81fbb02df92bc61afece,8.0000,-29.8000,30.5000,53.6000,0.3000,0.4000,0.4000,1.0000
0x5f612351a9e46afbe3f164473a7d77a69ec9840a,25.9000,38.2000,7.1000,41.8000,0.4000,0.4000,0.5000,1.0000
0xf10e941dbd215b98b90be7dd9ec61da409ef4308,9.9000,9.8000,28.4000,35.7000,0.0000,0.0000,0.1000,1.0000
0x04b3f873b003eba3774df1e5e47a370d27ee1b7e,4.1000,-8.7000,28.1000,16.3000,0.1000,0.2000,0.3000,1.0000


In [19]:
sim.to_csv(OUT_DIR / "price_scale_sim.csv", index=False)
summary.round(4).to_csv(OUT_DIR / "price_scale_summary.csv", index=False)
pw.round(2).reset_index().to_csv(OUT_DIR / "price_scale_wallets.csv", index=False)
print("saved -> signal_lab/price_scale_{sim,summary,wallets}.csv")


saved -> signal_lab/price_scale_{sim,summary,wallets}.csv
